# ⚠️ TEST — Project 1 applied to freMTPL2 (French motor TPL)

**Experimental public-data application.** The Project 1 pricing pipeline (frequency GLM + severity GLM -> pure premium, calibration checks, fairness-style audit) applied to real motor insurance data.

* **Data:** freMTPL2 (French Motor Third-Party Liability), public academic dataset; CSV mirror from https://huggingface.co/datasets/mabilton/fremtpl2
* **Committed sample:** 100K of the 678K policies (full file via `data/download_data.py`).
* **Protected proxy:** age band (`DrivAge`) — age discrimination in pricing is regulated in several jurisdictions.
* **Note:** this is a *demonstration*, not production pricing: naive features, no validation split, sample data.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT.parent))  # repo root for src.models / src.fairness

from src.models import predict_frequency
from src.fairness import base_rates, calibration_by_group, demographic_parity

plt.rcParams["figure.dpi"] = 110
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

freq = pd.read_csv(ROOT / "data" / "freMTPL2freq_sample.csv")
sev = pd.read_csv(ROOT / "data" / "freMTPL2sev.csv")
print("freq:", freq.shape, "| sev:", sev.shape)
freq.head()

freq: (100000, 12) | sev: (26639, 2)


,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,15.0,1,0.45,6,2,38,50,B12,Regular,E,3003,Nord-Pas-de-Calais
1,17.0,1,0.27,7,0,33,68,B12,Diesel,C,137,Languedoc-Roussillon
2,21.0,1,0.15,7,0,41,50,B12,Diesel,B,60,Pays-de-la-Loire
3,35.0,1,0.76,4,9,23,100,B6,Regular,E,7887,Nord-Pas-de-Calais
4,77.0,1,0.69,6,0,60,51,B12,Diesel,A,12,Auvergne


## Prepare the pricing frame

* Join claim amounts to policies (0 if no claim).
* `pure premium = expected frequency x expected severity per claim`.
* Add an age-band column as the protected proxy.

In [2]:
sev_total = sev.groupby("IDpol", as_index=False)["ClaimAmount"].sum()
df = freq.merge(sev_total, on="IDpol", how="left")
df["ClaimAmount"] = df["ClaimAmount"].fillna(0.0)
df["DrivAgeBand"] = pd.cut(
    df["DrivAge"], bins=[17, 25, 40, 60, 120], right=False,
    labels=["<25", "25-40", "40-60", "60+"],
)
df["logDensity"] = np.log(df["Density"])
df["policy_id"] = df["IDpol"]
print(f"policies: {len(df):,} | claim rate: {(df.ClaimNb > 0).mean():.4f} | mean loss: {df.ClaimAmount.mean():.2f}")
df.head()

def outcome_rates(groups):
    tab = df.groupby(list(groups), observed=True).agg(
        n_policies=("IDpol", "count"),
        positive_rate=("ClaimNb", lambda s: (s > 0).mean()),
        mean_frequency=("ClaimNb", "mean"),
    )
    sev = df.loc[df["ClaimNb"] > 0].groupby(list(groups), observed=True)["ClaimAmount"].mean().rename("avg_severity_given_claim")
    return tab.join(sev).round(4)

policies: 100,000 | claim rate: 0.0500 | mean loss: 79.07


## Frequency GLM (Poisson, exposure offset)

Same structure as Project 1: claim counts with `log(Exposure)` as offset.

In [3]:
freq_formula = ("ClaimNb ~ DrivAge + VehAge + VehPower + BonusMalus + logDensity + "
                 "C(Area) + C(VehBrand) + C(VehGas)")
freq_model = smf.glm(freq_formula, data=df, family=sm.families.Poisson(),
                     offset=np.log(df["Exposure"])).fit()
print(freq_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               100000
Model:                            GLM   Df Residuals:                    99978
Model Family:                 Poisson   Df Model:                           21
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -21114.
Date:                Mon, 10 Aug 2026   Deviance:                       32050.
Time:                        01:17:00   Pearson chi2:                 2.76e+05
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01030
Covariance Type:            nonrobust                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -3.8898 

In [4]:
# Frequency calibration check: mean predicted vs observed claim count
pred_freq = predict_frequency(freq_model, df, exposure_col="Exposure")
print(f"mean predicted ClaimNb: {pred_freq.mean():.4f} | observed: {df['ClaimNb'].mean():.4f}")

mean predicted ClaimNb: 0.0531 | observed: 0.0531


## Severity GLM (Gamma, per claim)

Fitted on policies with at least one claim; response = `ClaimAmount / ClaimNb` (severity per claim, matching the Project 1 decomposition).

In [5]:
claimers = df[df["ClaimNb"] > 0].copy()
claimers["sev_per_claim"] = claimers["ClaimAmount"] / claimers["ClaimNb"]
sev_formula = ("sev_per_claim ~ DrivAge + VehAge + VehPower + BonusMalus + logDensity + "
               "C(Area) + C(VehBrand) + C(VehGas)")
sev_model = smf.glm(sev_formula, data=claimers,
                    family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print(sev_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          sev_per_claim   No. Observations:                 4997
Model:                            GLM   Df Residuals:                     4975
Model Family:                   Gamma   Df Model:                           21
Link Function:                    Log   Scale:                          14.463
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Mon, 10 Aug 2026   Deviance:                       94154.
Time:                        01:17:01   Pearson chi2:                 7.20e+04
No. Iterations:                    29   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                7.0874 

C:\Users\29226\Desktop\Github\EXP-project-1\.venv\Lib\site-packages\statsmodels\genmod\families\family.py:812: RuntimeWarning: divide by zero encountered in log
  ll_obs -= special.gammaln(weight_scale) + np.log(endog)
C:\Users\29226\Desktop\Github\EXP-project-1\.venv\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1891: RuntimeWarning: invalid value encountered in scalar subtract
  prsq = 1 - np.exp((self.llnull - self.llf) * (2 / self.nobs))


## Pure premium and calibration

In [6]:
df["predicted_premium"] = pred_freq * sev_model.predict(df)
cal = calibration_by_group(df, "ClaimAmount", "predicted_premium", ("Area",))
print(cal.to_string())

      n_policies  actual_mean  predicted_mean  predicted/actual
Area                                                           
A          15310      82.6747         74.9701            0.9068
B          11084      60.0711         57.7669            0.9616
C          28245      68.1235         70.5570            1.0357
D          22458      83.8332         81.0264            0.9665
E          20195      98.2119         94.6372            0.9636
F           2708      68.2148         54.0468            0.7923


In [7]:
cal_age = calibration_by_group(df, "ClaimAmount", "predicted_premium", ("DrivAgeBand",))
print(cal_age.to_string())

             n_policies  actual_mean  predicted_mean  predicted/actual
DrivAgeBand                                                           
<25                4422     192.1269        126.5055            0.6584
25-40             33426      75.7292         82.6591            1.0915
40-60             45531      73.5700         68.9524            0.9372
60+               16621      70.7505         71.9791            1.0174


## Fairness-style audit by age band

Age band is treated as a protected proxy. Base rates differ sharply (young drivers claim far more), which is exactly the situation where fairness constraints have a price.

In [8]:
outcome_rates(("DrivAgeBand",))

,n_policies,positive_rate,mean_frequency,avg_severity_given_claim
DrivAgeBand,,,,
<25,4422,0.0778,0.0837,2469.7247
25-40,33426,0.0439,0.0465,1723.1611
40-60,45531,0.0503,0.0533,1464.0360
60+,16621,0.0539,0.0575,1312.4382


In [9]:
demographic_parity(df, "predicted_premium", ("DrivAgeBand",))

,n_policies,mean_score,ratio_vs_overall,diff_vs_overall
DrivAgeBand,,,,
<25,4422,126.5055,1.6519,49.9234
25-40,33426,82.6591,1.0794,6.0770
40-60,45531,68.9524,0.9004,-7.6297
60+,16621,71.9791,0.9399,-4.6030


## Save results

In [10]:
cal.to_csv(RESULTS / "fremtpl2_calibration_area.csv")
cal_age.to_csv(RESULTS / "fremtpl2_calibration_ageband.csv")
outcome_rates(("DrivAgeBand",)).to_csv(RESULTS / "fremtpl2_base_rates_ageband.csv")
demographic_parity(df, "predicted_premium", ("DrivAgeBand",)).to_csv(RESULTS / "fremtpl2_dp_ageband.csv")

tab = df.groupby("DrivAgeBand", observed=True)["predicted_premium"].mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
tab.plot.bar(ax=ax, color="#4C72B0", edgecolor="white")
ax.set_ylabel("Predicted pure premium (EUR)")
ax.set_title("Predicted pure premium by age band (TEST, freMTPL2)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS / "fremtpl2_premium_by_ageband.png", dpi=150)
plt.close()
print("saved results to", RESULTS)

saved results to C:\Users\29226\Desktop\Github\EXP-project-1\test_public_examples\results


## Observations (TEST)

* The Poisson frequency model reproduces the observed claim count on average (calibration by construction) and the gamma severity model is roughly calibrated by region/age band.
* Base rates differ strongly by age band (young drivers ≈ 2-3x claim rate of older drivers), and the predicted premium spread across age bands is the "who pays more" story.
* **Caveats:** no train/test split, no bonus-malus or feature engineering care, protected proxy chosen for demonstration, and this is a sample of the full data. The point is pipeline transferability.